# MICA benchmark: scoring session `scoring_cryozeta_d`

Scores every model that the other notebooks built (CryoZeta from `gpu_cryozeta_d` (the third full session; added to the benchmark after Prof. Cheng's request of 2026-10-31); the other methods are scored in another session)
against the deposited models from RCSB, in one way for every method (the scoring protocol of LOG.md, 2026-10-24):

* US-align `-mm 1` (multi-chain; TM-score normalised by the deposited model, aligned length, RMSD, sequence identity),
* US-align `-mm 5` on one chain per structure (the fragment-agnostic TM-score),
* Phenix `chain_comparison` on the model superposed onto the deposited one and in its own frame (Calpha match, sequence match, with the coverage the Calpha quality score),
* the same counts from a nearest-neighbour search (a check, and the fallback if Phenix cannot be used).

The notebook needs **no AlphaFold file**. It writes `scores_long.csv` (one line per model), the status tables of the notebooks it read, the raw output of every program and a zip. The zip holds no coordinates.

**What you do (Kaggle, background mode)**: accelerator **None**, Internet **On**, the two Phenix secrets (`PHENIX_USER`, `PHENIX_PASSWORD`) attached, and **the outputs of the other notebooks as inputs**:
*Add Input -> Notebook Output Files* (the tab with your own notebooks) -> add `gpu_cryozeta_d` **and nothing else** (the names you gave them; only the
versions that *Succeeded*; not the CPU notebooks). Then *Save Version -> Save & Run All (Commit)*. About one hour. When the version shows *Succeeded*, download `scoring_cryozeta_d_result_....zip` from its Output and send it to me.
**Do not stop or cancel the run**: a stopped run keeps nothing.

**Phenix** is licensed to you personally. Do not share the installer, the password, or this notebook with the secrets attached.


## Settings

In [ ]:
# WORKERS = models scored at the same time (a Kaggle CPU session has 4 cores).
WORKERS = 4


## Step 1. Look at the machine

In [ ]:
import csv, datetime, getpass, glob, gzip, hashlib, json, os, platform, re, shutil, signal, subprocess, sys, tarfile, threading, time, zipfile
from collections import deque

IN_KAGGLE = os.path.isdir("/kaggle/working") and not os.environ.get("MICA_WORK")      # MICA_WORK and MICA_BIG are only used by the tests
IN_COLAB = (not IN_KAGGLE) and os.path.isdir("/content") and not os.environ.get("MICA_WORK")
PLATFORM = "kaggle" if IN_KAGGLE else ("colab" if IN_COLAB else "other")
if os.environ.get("MICA_WORK"):
    WORK = OUT_DIR = os.environ["MICA_WORK"]
elif IN_KAGGLE:
    WORK, OUT_DIR = "/kaggle/working/pilot", "/kaggle/working"
elif IN_COLAB:
    WORK, OUT_DIR = "/content/pilot", "/content"
else:
    WORK = OUT_DIR = os.path.abspath("pilot")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
LOG = os.path.join(WORK, "run.log")
REQUIRE_GPU = False
NET_HOSTS = (('PyPI', 'https://pypi.org/simple/uv/', True), ('GitHub', 'https://github.com/jianlin-cheng/MICA', True), ('RCSB', 'https://files.rcsb.org/header/26DK.cif', True), ('Phenix', 'https://phenix-online.org/download', True))
STATE = {"ok": True, "problem": "", "t0": time.time(), "started": datetime.datetime.now(datetime.timezone.utc).isoformat(),
         "phenix_env": None, "phenix_version": "", "installer_sha256": "", "usalign": "", "usalign_version": "", "weights": {}, "env": {},
         "entries": {}, "gpu": ""}
SECRETS = []          # kept in memory only, to make sure they never end up in the zip


def log(msg=""):
    print(msg, flush=True)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")


def fail(msg):
    STATE["ok"] = False
    STATE["problem"] = msg
    log("\n" + "!" * 72 + "\nSTOPPED: " + msg + "\nThe steps below are skipped. The last step still packs a zip with the log: send it to me.\n" + "!" * 72)


def run(cmd, env=None, every=1, cwd=None, log_file=None, timeout=None, quiet=False):
    """Runs a command, prints and logs its output as it comes. Returns (exit code, last lines, timed_out).
    env: values are set, None removes the variable. A command that runs longer than `timeout` seconds is killed with its child processes.
    Progress bars (lines with %|) are printed only now and then; quiet=True keeps the output out of the notebook (it still goes to log_file)."""
    e = dict(os.environ)
    for k, v in (env or {}).items():
        if v is None:
            e.pop(k, None)
        else:
            e[k] = v
    e["PYTHONUNBUFFERED"] = "1"
    try:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e, cwd=cwd, errors="replace",
                             start_new_session=True)
    except OSError as ex:
        log("could not start %s: %s" % (cmd[0] if isinstance(cmd, (list, tuple)) else cmd, ex))
        return 127, [str(ex)], False
    timed = {"out": False}

    def killer():
        timed["out"] = True
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except OSError:
            pass

    timer = threading.Timer(timeout, killer) if timeout else None
    if timer:
        timer.start()
    lf = open(log_file, "a", encoding="utf-8") if log_file else None
    last, n, bars = deque(maxlen=40), 0, 0
    try:
        for line in p.stdout:
            line = line.rstrip("\n")
            if len(line) > 600:
                line = line[:600] + " ..."
            last.append(line)
            n += 1
            if lf:
                lf.write(line + "\n")
                lf.flush()                                  # a program that is silent for an hour must not hide its last lines in a buffer
            if "%|" in line:
                bars += 1
                show = bars % 100 == 1
            else:
                show = every == 1 or n <= 5 or n % every == 0
            if show and not quiet:
                log(line)
        p.wait()
    finally:
        if timer:
            timer.cancel()
        if lf:
            lf.close()
        p.stdout.close()
    return p.returncode, list(last), timed["out"]


def sh(cmd, timeout=60):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, errors="replace")
        return (r.stdout + r.stderr).strip()
    except Exception as ex:
        return "ERROR: %s" % ex


def secret(name, prompt, hidden=True):
    """Colab Secrets, Kaggle Secrets, an environment variable, or a prompt, in this order. Returns "" if nothing can be found."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    try:
        return (getpass.getpass(prompt) if hidden else input(prompt)).strip()
    except Exception:
        return ""


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def meminfo_gb():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    return round(int(line.split()[1]) / 1048576, 1)
    except OSError:
        pass
    return None


def pick_big_dir():
    """The folder for everything large (Phenix, Python environment, MICA): the disk with the most free space."""
    if os.environ.get("MICA_BIG"):
        return os.environ["MICA_BIG"]
    best = None
    for base in ("/kaggle/temp", "/tmp", "/content", WORK):
        if os.path.isdir(base) and os.access(base, os.W_OK):
            free = shutil.disk_usage(base).free
            if best is None or free > best[1] + 5e9:
                best = (base, free)
    return os.path.join(best[0], "mica_pilot")


BIG = pick_big_dir()
os.makedirs(BIG, exist_ok=True)


class Monitor(threading.Thread):
    """Samples GPU memory, GPU load and free RAM every few seconds into a csv file; summary() gives the peaks."""

    def __init__(self, path, every=5):
        super().__init__(daemon=True)
        self.path, self.every, self.stop_flag = path, every, threading.Event()
        self.peak_gpu_mb, self.peak_util, self.min_ram_gb, self.samples = 0, 0, None, 0

    def sample(self):
        gpu_mb, util = "", ""
        if shutil.which("nvidia-smi"):
            out = sh("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits", timeout=20).splitlines()
            try:
                a, b = [x.strip() for x in out[0].split(",")]
                gpu_mb, util = int(a), int(b)
                self.peak_gpu_mb, self.peak_util = max(self.peak_gpu_mb, gpu_mb), max(self.peak_util, util)
            except (ValueError, IndexError):
                gpu_mb, util = "", ""
        ram = ""
        try:
            with open("/proc/meminfo") as f:
                for line in f:
                    if line.startswith("MemAvailable"):
                        ram = round(int(line.split()[1]) / 1048576, 2)
        except OSError:
            pass
        if ram != "":
            self.min_ram_gb = ram if self.min_ram_gb is None else min(self.min_ram_gb, ram)
        self.samples += 1
        with open(self.path, "a", encoding="utf-8") as f:
            f.write("%s,%s,%s,%s\n" % (datetime.datetime.now(datetime.timezone.utc).strftime("%H:%M:%S"), gpu_mb, util, ram))

    def run(self):
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        with open(self.path, "w", encoding="utf-8") as f:
            f.write("time_utc,gpu_mem_used_mb,gpu_util_pct,ram_available_gb\n")
        while not self.stop_flag.is_set():
            try:
                self.sample()
            except Exception:
                pass
            self.stop_flag.wait(self.every)

    def stop(self):
        self.stop_flag.set()
        self.join(timeout=30)
        return {"peak_gpu_mb": self.peak_gpu_mb, "peak_gpu_util_pct": self.peak_util, "min_ram_available_gb": self.min_ram_gb, "samples": self.samples}


def probe_site(url, timeout=45):
    import urllib.error, urllib.request
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "mica-benchmark-pilot"}), timeout=timeout) as r:
            r.read(512)
            return True, "HTTP %s" % r.status
    except urllib.error.HTTPError as ex:           # the server answered (404 or 403 for a probe address): the way to the site is open; only 5xx counts as down
        return ex.code < 500, "HTTP %s" % ex.code
    except Exception as ex:                        # no answer: DNS, TLS handshake or connect time-out, connection reset, internet off
        return False, str(ex)[:100]


def check_internet(tries=4, pause=20):
    # Up to 4 rounds (20, 40 and 60 s apart), and only the sites that failed are asked again: one TLS time-out of one site (the EBI map server on
    # 7 November 2026) must not stop a session of 11 hours. A session with the internet off still stops after about 2 minutes, before any hour is spent.
    pending, last = list(NET_HOSTS), {}
    for rnd in range(1, tries + 1):
        retry = []
        for host in pending:
            ok, text = probe_site(host[1])
            if ok:
                log("  reachable: %-9s %s%s" % (host[0], text, "" if rnd == 1 else " (at try %d)" % rnd))
            else:
                last[host[0]] = text
                retry.append(host)
                log("  no answer (try %d of %d): %-9s %s" % (rnd, tries, host[0], text))
        pending = retry
        if not pending or rnd == tries:
            break
        time.sleep(pause * rnd)
    for name, url, required in pending:
        log("  NOT reachable: %-9s %s" % (name, "needed: the notebook stops" if required else "optional: the notebook goes on"))
    return [name for name, url, required in pending if required]


free_big, free_work = round(shutil.disk_usage(BIG).free / 1e9), round(shutil.disk_usage(WORK).free / 1e9)
cpu_model = sh("lscpu | grep -m1 'Model name'").split(":")[-1].strip()
log("platform %s | python %s | %s | %s CPU cores (%s) | %s GB RAM" % (PLATFORM, platform.python_version(), platform.platform(), os.cpu_count(), cpu_model, meminfo_gb()))
log("large files go to %s (%d GB free); results to %s (%d GB free)" % (BIG, free_big, WORK, free_work))
with open("machine.txt", "w", encoding="utf-8") as f:
    f.write(sh("nvidia-smi") + "\n\n" + sh("lscpu") + "\n\n" + sh("free -h") + "\n\n" + sh("df -h") + "\n\n" + sh("g++ --version | head -1; git --version; uname -a; cat /etc/os-release | head -3") + "\n")
STATE["gpu"] = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") if shutil.which("nvidia-smi") else ""
if STATE["gpu"] and not STATE["gpu"].startswith("ERROR"):
    log("GPU: " + STATE["gpu"].replace("\n", " | "))
elif REQUIRE_GPU:
    fail("this session has no GPU. Kaggle: open Session options on the right, set Accelerator to GPU T4 x2 (or GPU P100), then Run all again. "
         "Colab: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
if STATE["ok"]:
    log("checking the internet:")
    missing = check_internet() if NET_HOSTS else []
    if missing:                                    # a site that the notebook needs (required in NET_HOSTS): stop now, before any hour is spent
        fail("the internet is off, or a site that this notebook needs cannot be reached (%s). Kaggle: open Session options on the right and switch Internet on "
             "(the account must be phone-verified), then Run all again. If Internet is on, the site may be down: run again in a few minutes." % ", ".join(missing))


## Step 2. Write the scripts and the frozen data files

The files below are exact copies of the files in the repository (scripts, the AlphaFold Server job table and the hashes of the 26 predictions,
the entry list, the exact list of Python packages). The last cell of this step checks their hashes.

In [ ]:
import os
for d in ("scripts", "data/af3", "env", "logs", "gpu"):
    os.makedirs(d, exist_ok=True)
print("folders ready")


In [ ]:
%%writefile scripts/nbtools.py
#!/usr/bin/env python3
"""Tools that the Phase 4 notebooks share. They were added after the first MICA pilot, which showed three weaknesses:

1. a program that is silent for an hour (Phenix docking) looked like a frozen notebook: `Heartbeat` prints one line every few minutes
   (elapsed time, load, free RAM, the busiest processes, the last line of the program's own log);
2. the zip with the results was only written by the very last cell, so a stopped session left nothing: `pack` writes the zip in
   one step and the notebooks call it after every stage (a fixed name, replaced each time);
3. the last cell used variables of earlier cells and failed after the kernel was restarted: `recover` needs nothing from the notebook,
   it finds the work folder on disk and packs whatever is there.

Everything the zip should hold is copied into the work folder by the notebook, so that these functions only walk that folder.
Not packed: maps and other large data, files over 60 MB, the AlphaFold Server predictions of the user (they must stay private), and any file
that contains one of the secrets (the Phenix password).

    python scripts/nbtools.py --work <work folder> --out <folder for the zip> --notebook <name>      # the recovery path, by hand
"""
from __future__ import annotations

import argparse
import datetime
import json
import os
import subprocess
import sys
import threading
import time
import zipfile

MAX_FILE = 60 * 1000 * 1000
SKIP_DIRS = {"AF3_results", "af3_upload", "af3_unzipped", "grids", "AF3_encodings", "temp_maps", "native", "downloads", "scripts", "data", "env", "__pycache__"}
SKIP_SUFFIXES = (".map", ".mrc", ".ccp4", ".gz", ".part", ".tmp", ".zip")
WORK_CANDIDATES = ("/kaggle/working/pilot", "/content/pilot")


def utc_now():
    return datetime.datetime.now(datetime.timezone.utc)


def last_line(path, max_chars=160):
    """The last non-empty line of a text file, or "" if there is none or the file cannot be read."""
    try:
        with open(path, "rb") as f:
            f.seek(0, os.SEEK_END)
            size = f.tell()
            f.seek(max(0, size - 8192))
            tail = f.read().decode("utf-8", errors="replace")
    except OSError:
        return ""
    lines = [ln.strip() for ln in tail.splitlines() if ln.strip()]
    return lines[-1][:max_chars] if lines else ""


def machine_load():
    """(load average over 1 minute, free RAM in GB, text about the three busiest processes), each None or "" where it cannot be read."""
    load = free = None
    try:
        with open("/proc/loadavg") as f:
            load = float(f.read().split()[0])
    except (OSError, ValueError, IndexError):
        pass
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemAvailable"):
                    free = round(int(line.split()[1]) / 1048576, 1)
    except (OSError, ValueError, IndexError):
        pass
    top = ""
    try:
        r = subprocess.run(["ps", "-eo", "pcpu,etimes,comm", "--sort=-pcpu"], capture_output=True, text=True, timeout=10, errors="replace")
        rows = [ln.split(None, 2) for ln in r.stdout.splitlines()[1:4] if ln.strip()]
        top = ", ".join("%s %s%% (%ss)" % (c[2][:18], c[0], c[1]) for c in rows if len(c) == 3)
    except (OSError, subprocess.SubprocessError):
        pass
    return load, free, top


class Heartbeat(threading.Thread):
    """While a long program runs in the main thread, prints one line every `every` seconds through `log`. `watch` is a list of files whose last
    line is shown (the program's own log); `on_tick` is called after each line (the notebooks use it to refresh the partial zip)."""

    def __init__(self, label, log, watch=(), every=120, on_tick=None):
        super().__init__(daemon=True)
        self.label, self.log, self.watch, self.every, self.on_tick = label, log, list(watch), every, on_tick
        self.stop_flag, self.t0, self.ticks = threading.Event(), time.time(), 0

    def line(self):
        load, free, top = machine_load()
        minutes = (time.time() - self.t0) / 60
        parts = ["[%s] running for %.0f min" % (self.label, minutes) if minutes >= 1 else "[%s] running for %.0f s" % (self.label, time.time() - self.t0)]
        if load is not None:
            parts.append("load %.1f" % load)
        if free is not None:
            parts.append("free RAM %.1f GB" % free)
        if top:
            parts.append("busiest: " + top)
        for path in self.watch:
            if os.path.exists(path):
                age = time.time() - os.path.getmtime(path)
                parts.append("%s: %d bytes, changed %s ago, last line: %s" % (os.path.basename(path), os.path.getsize(path),
                                                                           "%.0f s" % age if age < 120 else "%.0f min" % (age / 60), last_line(path) or "(empty)"))
        return " | ".join(parts)

    def run(self):
        while not self.stop_flag.wait(self.every):
            try:
                self.log(self.line())
                self.ticks += 1
                if self.on_tick:
                    self.on_tick()
            except Exception as ex:                       # a failing heartbeat must never stop the program it watches
                try:
                    self.log("[%s] heartbeat problem: %s" % (self.label, str(ex)[:150]))
                except Exception:
                    pass

    def stop(self):
        self.stop_flag.set()
        self.join(timeout=30)


def collect(work, skip_dirs=SKIP_DIRS, skip_suffixes=SKIP_SUFFIXES):
    """All files below `work` that may go into the zip, in a fixed order."""
    files = []
    for dirpath, dirs, names in os.walk(work):
        dirs[:] = sorted(d for d in dirs if d not in skip_dirs)
        for n in sorted(names):
            if not n.endswith(skip_suffixes):
                files.append(os.path.join(dirpath, n))
    return files


def pack(work, zip_path, root_name, secrets=(), max_file=MAX_FILE, extra=()):
    """Writes `zip_path` in one step (a temporary file that replaces the old zip at the end, so a stop in the middle never leaves a broken zip).
    Returns (number of files, list of what was left out and why). `extra` are files outside `work` that go in as well."""
    wanted = collect(work) + [p for p in extra if os.path.isfile(p)]
    left_out, tmp = [], zip_path + ".tmp"
    try:
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
            for path in dict.fromkeys(wanted):
                try:
                    if os.path.getsize(path) > max_file:
                        left_out.append("%s (larger than %d MB)" % (os.path.relpath(path, work), max_file // 1000000))
                        continue
                    with open(path, "rb") as f:
                        data = f.read()
                except OSError as ex:
                    left_out.append("%s (%s)" % (os.path.relpath(path, work), str(ex)[:60]))
                    continue
                if any(s and s.encode() in data for s in secrets):
                    left_out.append("%s (contained the password)" % os.path.relpath(path, work))
                    continue
                inside = os.path.relpath(path, work) if path.startswith(work + os.sep) else os.path.join("outside", path.lstrip(os.sep))
                z.writestr(root_name + "/" + inside, data)
            n = len(z.namelist())
        os.replace(tmp, zip_path)
    except BaseException:
        try:
            os.remove(tmp)
        except OSError:
            pass
        raise
    return n, left_out


def partial_zip_path(out_dir, notebook):
    return os.path.join(out_dir, "%s_partial.zip" % notebook)


def final_pack(work, out_dir, notebook, complete, secrets=(), extra=(), note=""):
    """The zip that is sent to me. complete: True (everything ran), False (something stopped) or None (unknown: recovered after a restart).
    The partial zip is removed once this one exists."""
    stamp = utc_now().strftime("%Y%m%d-%H%M")
    suffix = "" if complete else ("_INCOMPLETE" if complete is False else "_RECOVERED")
    name = "%s_result_%s%s" % (notebook, stamp, suffix)
    path = os.path.join(out_dir, name + ".zip")
    n, left_out = pack(work, path, name, secrets=secrets, extra=extra)
    try:
        os.remove(partial_zip_path(out_dir, notebook))
    except OSError:
        pass
    return path, n, left_out


def find_work(env=None):
    """The work folder of a notebook that ran in this session: the one with a run.log."""
    env = os.environ if env is None else env
    for cand in (env.get("MICA_WORK"),) + WORK_CANDIDATES + (os.path.abspath("pilot"),):
        if cand and os.path.isfile(os.path.join(cand, "run.log")):
            return cand
    return None


def known_secrets(env=None):
    """The Phenix password, if it can be read again (environment, Kaggle or Colab secrets), to check the files for it after a restart."""
    env = os.environ if env is None else env
    found = []
    if env.get("PHENIX_PASSWORD"):
        found.append(env["PHENIX_PASSWORD"].strip())
    for getter in ("kaggle", "colab"):
        try:
            if getter == "kaggle":
                from kaggle_secrets import UserSecretsClient
                v = UserSecretsClient().get_secret("PHENIX_PASSWORD")
            else:
                from google.colab import userdata
                v = userdata.get("PHENIX_PASSWORD")
            if v:
                found.append(v.strip())
        except Exception:
            pass
    return [s for s in dict.fromkeys(found) if len(s) >= 6]


def read_state(work):
    try:
        with open(os.path.join(work, "state.json"), encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}


def recover(notebook, out_dir=None, env=None, say=print):
    """The last step when nothing from the notebook is left in memory (kernel restarted, cell run alone, ...). Returns the zip path or None."""
    env = os.environ if env is None else env
    work = find_work(env)
    if not work:
        say("RECOVERY: no work folder with a run.log was found (looked in %s). Nothing to pack." % ", ".join(WORK_CANDIDATES))
        return None
    out_dir = out_dir or env.get("MICA_WORK") or os.path.dirname(work)
    state = read_state(work)
    info = {"notebook": notebook, "recovered": True, "recovered_utc": utc_now().isoformat(), "state_json_found": bool(state), "state": state}
    with open(os.path.join(work, "run_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, default=str)
    path, n, left_out = final_pack(work, out_dir, notebook, complete=None, secrets=known_secrets(env))
    say("RECOVERY: zip written from the files that are on disk: %s (%d files, %d MB)" % (path, n, os.path.getsize(path) // 1000000))
    if left_out:
        say("left out of the zip: %s" % "; ".join(left_out))
    say("If a notebook cell stopped with an error, send me this zip and the text of the error.")
    return path


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--work", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--notebook", required=True)
    args = p.parse_args(argv)
    path, n, left_out = final_pack(args.work, args.out, args.notebook, complete=None)
    print("%s (%d files)" % (path, n))
    for x in left_out:
        print("left out:", x)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile scripts/score_models.py
#!/usr/bin/env python3
"""Scores models against deposited models: protocol v1.0 of LOG.md (2026-10-24, "Scoring protocol").

Every model (the prediction) is compared with the deposited model of its entry (the native) in the same way for every method:

* US-align `-mm 1 -ter 1 -mol protein` (prediction = structure 1): tm_score (TM2, normalised by the deposited model), aligned_ca_length,
  ca_rmsd, seq_identity; tm_score_pred_norm (TM1) for information. The rotation matrix of this run puts the prediction into the frame of the deposited model.
* US-align `-mm 5` on one chain per structure (all chains of the prediction concatenated, all chains of the deposited model concatenated): the fragment-agnostic
  TM-score (tm_score_fns, ...). Fragment order, chain breaks and extra copies do not matter.
* `phenix.chain_comparison <native> <prediction>` on the prediction superposed by the matrix above (columns without a suffix) and on the prediction in its own
  frame (suffix _ff): ca_match = 100 * CLOSE / (CLOSE + FAR), ca_found = % FOUND, seq_match = SEQ MATCH(%), ca_rmsd_close; coverage = residues of the prediction /
  residues of the deposited model; ca_quality = ca_match * coverage.
* The same counts from a nearest-neighbour search (own_*): the fallback when Phenix cannot be used, and a check of the parsing.

All files that are handed to US-align (mm 5) and to Phenix are C-alpha traces written by this module (protein residues only, first model, first alternative
location), so that the formats and the extra atoms of the different methods cannot matter. The original file of the prediction is read by US-align for the
`-mm 1` run, as in the paper.

    python scripts/score_models.py one --usalign USalign --native 26DK.cif --pred model.pdb [--phenix-env env.sh] --workdir out/
    python scripts/score_models.py batch --usalign USalign --roots result1/ result2/ --entries data/entries_2026.csv --natives natives/ --out scores/ [--phenix-env env.sh]
"""
from __future__ import annotations

import argparse
import csv
import glob
import json
import math
import os
import re
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

PROTOCOL_VERSION = "1.0"
CUTOFF = 3.0                 # phenix.chain_comparison max_dist (its default) and the cut-off of the own counts
USALIGN_TIMEOUT = 900
PHENIX_TIMEOUT = 900
CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
PRIMARY_KINDS = {"modelangelo", "e3_prealign", "emprot", "emodelx_af_all_atom", "mica_all_atom", "cryoatom1", "cryozeta"}
METHOD_OF_KIND = {"modelangelo": "modelangelo", "modelangelo_raw": "modelangelo", "e3_prealign": "e3", "emprot": "emprot", "emprot_denovo": "emprot", "emprot_fit": "emprot",
                  "emodelx_af_all_atom": "emodelx", "emodelx_af_ca": "emodelx", "cryoatom1": "cryoatom1", "cryoatom1_raw": "cryoatom1", "cryozeta": "cryozeta"}
MODEL_SUFFIXES = (".pdb", ".cif", ".ent", ".mmcif")
CC_COLUMNS = ["rmsd", "close_n", "far_n", "forward", "reverse", "mixed", "found_pct", "ca_score", "seq_match_pct", "seq_score", "mean_length", "fragments", "bad_connections"]

SCORE_COLUMNS = (["entry", "method", "model_kind", "primary", "model_file", "source", "status", "note", "pred_residues", "native_residues", "coverage",
                  "n_chains_pred", "n_chains_native", "n_chains_paired", "longest_chain_pred", "us_residues_pred",
                  "tm_score", "tm_score_pred_norm", "ca_rmsd", "aligned_ca_length", "seq_identity",
                  "tm_score_fns", "ca_rmsd_fns", "aligned_ca_length_fns", "seq_identity_fns",
                  "cc_status", "ca_match", "ca_found", "seq_match", "ca_rmsd_close", "ca_quality", "cc_close_n", "cc_far_n",
                  "cc_status_ff", "ca_match_ff", "ca_found_ff", "seq_match_ff", "ca_rmsd_close_ff", "ca_quality_ff",
                  "own_ca_match", "own_ca_found", "own_seq_match", "own_ca_match_ff", "own_ca_found_ff", "own_seq_match_ff", "seconds"])


# ---------------------------------------------------------------------------------------------------------------------------------- C-alpha traces

class Trace:
    """The C-alpha trace of a structure: a list of chains, each with its name, its residues (name, number, insertion code) and an (n, 3) array of coordinates."""

    def __init__(self, chains=None):
        self.chains = chains or []

    @property
    def n_residues(self):
        return sum(len(c["res"]) for c in self.chains)

    @property
    def n_chains(self):
        return len(self.chains)

    def xyz(self):
        return np.concatenate([c["xyz"] for c in self.chains]) if self.chains else np.zeros((0, 3))

    def names(self):
        return [r[0] for c in self.chains for r in c["res"]]

    def transformed(self, matrix):
        """x' = t + U x for every residue; `matrix` is the (t, U) of parse_matrix."""
        t, u = matrix
        return Trace([{"name": c["name"], "res": list(c["res"]), "xyz": c["xyz"] @ u.T + t} for c in self.chains])

    def merged(self):
        """All residues in one chain A, numbered 1 to n (the order of the file)."""
        names = self.names()
        if not names:
            return Trace([])
        return Trace([{"name": "A", "res": [(n, i + 1, " ") for i, n in enumerate(names)], "xyz": self.xyz().copy()}])


def is_amino_acid(name):
    import gemmi
    name = name.strip().upper()
    info = gemmi.find_tabulated_residue(name)
    if info is not None:
        return bool(info.is_amino_acid()) or name == "UNK"
    return name == "UNK"


def read_trace(path):
    """The C-alpha trace of the first model of a pdb or mmCIF file (gemmi): amino-acid residues that have a CA atom, first alternative location."""
    import gemmi
    st = gemmi.read_structure(path)
    if len(st) == 0:
        raise ValueError("the file has no model")
    chains = []
    for ch in st[0]:
        res, xyz = [], []
        for r in ch:
            if not is_amino_acid(r.name):
                continue
            a = r.find_atom("CA", "*")
            if a is None:
                continue
            icode = (r.seqid.icode or " ").strip() or " "
            res.append((r.name.strip(), int(r.seqid.num), icode))
            xyz.append((a.pos.x, a.pos.y, a.pos.z))
        if res:
            chains.append({"name": ch.name, "res": res, "xyz": np.array(xyz, dtype=float)})
    return Trace(chains)


def chain_labels(trace):
    names = [c["name"] for c in trace.chains]
    if all(len(n) == 1 for n in names) and len(set(names)) == len(names):
        return names
    return [CHAIN_IDS[i % len(CHAIN_IDS)] for i in range(len(names))]


def write_pdb(trace, path):
    """A PDB file with the C-alpha atoms of the trace. Chain names are kept if they are single characters and unique, otherwise they are A, B, C, ... in order."""
    labels = chain_labels(trace)
    lines, serial = [], 0
    for lab, c in zip(labels, trace.chains):
        for (rn, num, ic), (x, y, z) in zip(c["res"], c["xyz"]):
            if not -999 <= num <= 9999:
                raise ValueError("residue number %d does not fit the PDB format" % num)
            serial += 1
            lines.append("ATOM  %5d  CA  %3s %1s%4d%1s   %8.3f%8.3f%8.3f%6.2f%6.2f          %2s" % (serial % 100000, rn[:3], lab, num, ic, x, y, z, 1.0, 20.0, "C"))
        lines.append("TER")
    lines.append("END")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    return path


# ---------------------------------------------------------------------------------------------------------------------------------- US-align

TABLE_COLUMNS = ["PDBchain1", "PDBchain2", "TM1", "TM2", "RMSD", "ID1", "ID2", "IDali", "L1", "L2", "Lali"]


def parse_usalign_tabular(text):
    rows = []
    for line in text.splitlines():
        if not line.strip() or line.startswith("#"):
            continue
        parts = line.rstrip("\n").split("\t")
        if len(parts) != len(TABLE_COLUMNS):
            raise ValueError("a US-align line has %d fields, %d expected: %r" % (len(parts), len(TABLE_COLUMNS), line[:200]))
        row = dict(zip(TABLE_COLUMNS, parts))
        for k in ("TM1", "TM2", "RMSD", "ID1", "ID2", "IDali"):
            row[k] = float(row[k])
        for k in ("L1", "L2", "Lali"):
            row[k] = int(row[k])
        rows.append(row)
    return rows


def parse_matrix(text):
    """(t, U) of the rotation matrix file of US-align (`-m`): X = t + U x rotates structure 1 onto structure 2."""
    rows = []
    for line in text.splitlines():
        m = re.match(r"^\s*([012])\s+([-+]?\d+\.\d+)\s+([-+]?\d+\.\d+)\s+([-+]?\d+\.\d+)\s+([-+]?\d+\.\d+)\s*$", line)
        if m:
            rows.append([float(m.group(i)) for i in (2, 3, 4, 5)])
    if len(rows) != 3:
        raise ValueError("no rotation matrix in the US-align matrix file")
    a = np.array(rows)
    return a[:, 0].copy(), a[:, 1:].copy()


def chain_pairs(row):
    """Chain pairs of a `-mm 1` row: the i-th chain of structure 1 is paired with the i-th chain of structure 2 (empty names are unpaired chains)."""
    c1 = row["PDBchain1"].split(":")[1:]
    c2 = row["PDBchain2"].split(":")[1:]
    return [(a, b) for a, b in zip(c1, c2) if a and b]


def run_usalign(usalign, pred, native, mm, ter, workdir, tag, timeout=USALIGN_TIMEOUT):
    """One US-align run; returns the table row (a dict) with the matrix under "matrix"; raises RuntimeError with the reason."""
    matrix = os.path.join(workdir, tag + "_matrix.txt")
    if os.path.exists(matrix):
        os.remove(matrix)
    cmd = [usalign, pred, native, "-mol", "protein", "-mm", str(mm), "-ter", str(ter), "-outfmt", "2", "-m", matrix]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, errors="replace")
    except subprocess.TimeoutExpired:
        raise RuntimeError("US-align (-mm %d) did not finish within %d seconds" % (mm, timeout))
    except OSError as e:
        raise RuntimeError("US-align could not be started: %s" % e)
    with open(os.path.join(workdir, tag + "_usalign.txt"), "w", encoding="utf-8") as f:
        f.write("# %s\n%s%s" % (" ".join(cmd), r.stdout, ("\n" + r.stderr) if r.stderr else ""))
    if r.returncode != 0:
        raise RuntimeError("US-align (-mm %d) stopped with exit code %d: %s" % (mm, r.returncode, (r.stderr or r.stdout).strip()[-300:]))
    try:
        rows = parse_usalign_tabular(r.stdout)
    except ValueError as e:
        raise RuntimeError("US-align (-mm %d): %s" % (mm, e))
    if len(rows) != 1:
        raise RuntimeError("US-align (-mm %d) gave %d result lines, 1 expected. Output: %s" % (mm, len(rows), r.stdout.strip()[-300:]))
    row = rows[0]
    try:
        with open(matrix, encoding="utf-8", errors="replace") as f:
            row["matrix"] = parse_matrix(f.read())
    except (OSError, ValueError):
        row["matrix"] = None
    return row


# ---------------------------------------------------------------------------------------------------------------------------------- phenix.chain_comparison

_ROW = re.compile(r"^\s*([A-Za-z_][^\s]*)\s+((?:[-+]?\d+(?:\.\d+)?\s+){12}[-+]?\d+(?:\.\d+)?)\s*$")


def parse_chain_comparison(text):
    """The result row of phenix.chain_comparison (label and 13 numbers: RMSD, CLOSE N, FAR N, FORWARD, REVERSE, MIXED, % FOUND, CA SCORE, SEQ MATCH(%), SEQ SCORE,
    MEAN LENGTH, FRAGMENTS, BAD CONNECTIONS) as a dict, with the residues of the target and of the query that it used."""
    rows = [m for m in (_ROW.match(line) for line in text.splitlines()) if m]
    if not rows:
        raise ValueError("no result row in the output of phenix.chain_comparison")
    d = dict(zip(CC_COLUMNS, [float(x) for x in rows[0].group(2).split()]))
    d["label"] = rows[0].group(1)
    d["n_rows"] = len(rows)
    used = re.search(r"Used residues in target:\s*(\d+)\s+In query:\s*(\d+)", text)
    d["used_target"], d["used_query"] = (int(used.group(1)), int(used.group(2))) if used else (None, None)
    return d


def run_chain_comparison(phenix_env, target, query, out_txt, timeout=PHENIX_TIMEOUT):
    """phenix.chain_comparison target query (the target comes first, as in the Phenix documentation); the whole output is kept in out_txt."""
    cmd = ["bash", "-c", 'source "$0" && phenix.chain_comparison "$1" "$2"', phenix_env, target, query]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, errors="replace", cwd=os.path.dirname(os.path.abspath(out_txt)))
    except subprocess.TimeoutExpired:
        raise RuntimeError("phenix.chain_comparison did not finish within %d seconds" % timeout)
    text = r.stdout + (("\n" + r.stderr) if r.stderr else "")
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write("# phenix.chain_comparison %s %s\n%s" % (target, query, text))
    if r.returncode != 0:
        raise RuntimeError("phenix.chain_comparison stopped with exit code %d: %s" % (r.returncode, text.strip()[-300:]))
    try:
        return parse_chain_comparison(text)
    except ValueError as e:
        raise RuntimeError("%s: %s" % (e, text.strip()[-300:]))


def cc_metrics(d, coverage):
    """ca_match (percentage of the query residues within the cut-off of a deposited residue), ca_found, seq_match, rmsd of the close residues, ca_quality."""
    close, far = d["close_n"], d["far_n"]
    ca_match = 100.0 * close / (close + far) if close + far > 0 else 0.0
    out = {"ca_match": round(ca_match, 3), "ca_found": d["found_pct"], "seq_match": d["seq_match_pct"], "ca_rmsd_close": d["rmsd"],
           "cc_close_n": int(close), "cc_far_n": int(far)}
    out["ca_quality"] = round(ca_match * coverage, 3) if coverage != "" else ""
    return out


# ---------------------------------------------------------------------------------------------------------------------------------- own counts

def own_stats(query, target, cutoff=CUTOFF):
    """The counts of chain_comparison from a nearest-neighbour search: the share of the query residues within the cut-off of a target residue (ca_match), the share of the
    target residues within the cut-off of a query residue (ca_found) and the share of the close query residues whose residue name equals that of the nearest target residue."""
    from scipy.spatial import cKDTree
    q, t = query.xyz(), target.xyz()
    if len(q) == 0 or len(t) == 0:
        return {"ca_match": 0.0, "ca_found": 0.0, "seq_match": 0.0}
    d, idx = cKDTree(t).query(q)
    close = d <= cutoff
    dq, _ = cKDTree(q).query(t)
    qn, tn = query.names(), target.names()
    same = sum(1 for i in np.nonzero(close)[0] if qn[i] == tn[idx[i]])
    n_close = int(close.sum())
    return {"ca_match": round(100.0 * n_close / len(q), 3), "ca_found": round(100.0 * float((dq <= cutoff).sum()) / len(t), 3),
            "seq_match": round(100.0 * same / n_close, 3) if n_close else 0.0}


# ---------------------------------------------------------------------------------------------------------------------------------- one model

def _num(x, digits=4):
    return "" if x is None or x == "" else round(float(x), digits)


def score_model(usalign, phenix_env, native, pred_path, workdir, native_file=None):
    """All numbers of one model. `native` is the Trace of the deposited model and `native_file` its C-alpha file (written if it is None).
    Returns a dict with the keys of SCORE_COLUMNS that this function fills; the raw outputs of the programs are in workdir."""
    t0 = time.time()
    os.makedirs(workdir, exist_ok=True)
    out = {k: "" for k in SCORE_COLUMNS}
    out.update(status="ok", native_residues=native.n_residues, n_chains_native=native.n_chains)
    try:
        pred = read_trace(pred_path)
    except Exception as e:
        out.update(status="no_model", note=("the file cannot be read: %s" % e)[:300], seconds=round(time.time() - t0, 1))
        return out
    if pred.n_residues == 0:
        out.update(status="no_model", note="the file has no amino-acid residue with a CA atom", pred_residues=0, n_chains_pred=0, seconds=round(time.time() - t0, 1))
        return out
    if native_file is None:
        native_file = write_pdb(native, os.path.join(workdir, "native_ca.pdb"))
    coverage = round(pred.n_residues / float(native.n_residues), 4)
    out.update(pred_residues=pred.n_residues, coverage=coverage, n_chains_pred=pred.n_chains, longest_chain_pred=max(len(c["res"]) for c in pred.chains))
    notes = []
    # 1. US-align, multi-chain, as in the paper (the original file of the prediction)
    matrix = None
    try:
        row = run_usalign(usalign, pred_path, native_file, 1, 1, workdir, "mm1")
        out.update(tm_score=_num(row["TM2"]), tm_score_pred_norm=_num(row["TM1"]), ca_rmsd=_num(row["RMSD"], 2), aligned_ca_length=row["Lali"], seq_identity=_num(row["IDali"]),
                   n_chains_paired=len(chain_pairs(row)), us_residues_pred=row["L1"])
        matrix = row["matrix"]
        if matrix is None:
            notes.append("no rotation matrix: the superposed Phenix variant was not run")
        if row["L1"] != pred.n_residues:
            notes.append("US-align read %d residues, the trace has %d" % (row["L1"], pred.n_residues))
    except RuntimeError as e:
        out.update(status="no_alignment")
        notes.append(str(e))
    # 2. US-align, one chain per structure, fully non-sequential (fragment-agnostic)
    try:
        pm = write_pdb(pred.merged(), os.path.join(workdir, "pred_merged.pdb"))
        nm = write_pdb(native.merged(), os.path.join(workdir, "native_merged.pdb"))
        row = run_usalign(usalign, pm, nm, 5, 2, workdir, "fns")
        out.update(tm_score_fns=_num(row["TM2"]), ca_rmsd_fns=_num(row["RMSD"], 2), aligned_ca_length_fns=row["Lali"], seq_identity_fns=_num(row["IDali"]))
    except (RuntimeError, ValueError) as e:
        notes.append("fns: %s" % e)
    # 3. Chain comparison: superposed (matrix of the -mm 1 run) and in the own frame of the prediction
    variants = [("", pred.transformed(matrix) if matrix is not None else None, "superposed"), ("_ff", pred, "frame_fixed")]
    for suffix, trace, tag in variants:
        if trace is None:
            continue
        try:
            qfile = write_pdb(trace, os.path.join(workdir, "query_%s.pdb" % tag))
        except ValueError as e:
            notes.append("%s: %s" % (tag, e))
            continue
        own = own_stats(trace, native)
        out.update({"own_ca_match" + suffix: own["ca_match"], "own_ca_found" + suffix: own["ca_found"], "own_seq_match" + suffix: own["seq_match"]})
        if not phenix_env:
            out["cc_status" + suffix] = "not run"
            continue
        try:
            d = run_chain_comparison(phenix_env, native_file, qfile, os.path.join(workdir, "chain_comparison_%s.txt" % tag))
            m = cc_metrics(d, coverage)
            out.update({"ca_match" + suffix: m["ca_match"], "ca_found" + suffix: m["ca_found"], "seq_match" + suffix: m["seq_match"],
                        "ca_rmsd_close" + suffix: m["ca_rmsd_close"], "ca_quality" + suffix: m["ca_quality"], "cc_status" + suffix: "ok"})
            if suffix == "":
                out.update(cc_close_n=m["cc_close_n"], cc_far_n=m["cc_far_n"])
            if d["used_query"] is not None and d["used_query"] != pred.n_residues:
                notes.append("%s: Phenix used %d of %d residues of the prediction" % (tag, d["used_query"], pred.n_residues))
        except RuntimeError as e:
            out["cc_status" + suffix] = "failed"
            notes.append("%s: %s" % (tag, str(e)[:200]))
    out["note"] = " | ".join(n.replace("\n", " ") for n in notes)[:600]
    out["seconds"] = round(time.time() - t0, 1)
    return out


# ---------------------------------------------------------------------------------------------------------------------------------- many models

def model_kind(method, filename):
    n = os.path.basename(filename)
    low = n.lower()
    if method == "mica":
        if "real_space_refined" in low:
            return "mica_refined"
        if low.endswith("_all_atom_model.pdb"):
            return "mica_all_atom"
        if "(init)" in low:
            return "mica_init_ca"
        if low.endswith("_ca_model.pdb"):
            return "mica_ca"
        return "mica_other"
    if method == "modelangelo":
        return "modelangelo_raw" if "_raw" in low else "modelangelo"
    if method == "e3":
        return "e3_prealign" if "prealign" in low else "e3_other"
    if method == "emprot":
        if "denovo" in low:
            return "emprot_denovo"
        if "fit" in low:
            return "emprot_fit"
        return "emprot"
    if method == "emodelx":
        if low.endswith("_all_atom_model.pdb"):
            return "emodelx_af_all_atom"
        if low.endswith("_ca_model.pdb"):
            return "emodelx_af_ca"
        return "emodelx_other"
    if method == "cryoatom1":                                   # added 2026-11-01 (Prof. Cheng's request): <ENTRY>_cryoatom1.cif is the model, <ENTRY>_cryoatom1_raw.cif the one before pruning
        return "cryoatom1_raw" if low.endswith("_raw.cif") else "cryoatom1"
    if method == "cryozeta":                                    # CryoZeta-Final/<ENTRY>_sample_<i>.cif, ranked best first by the program: sample 0 is the model of the benchmark
        m = re.search(r"_sample_(\d+)\.cif$", low)
        if not m:
            return "cryozeta_other"
        return "cryozeta" if int(m.group(1)) == 0 else "cryozeta_s%d" % int(m.group(1))
    return method + "_other"


def collect_models(roots):
    """Models of result folders (`<root>/models/<method>/<ENTRY>/<file>`), oldest root first; a model that appears in several roots is taken from the last one.
    Returns (list of dicts, list of warnings)."""
    found, warnings = {}, []
    for root in roots:
        label = os.path.basename(os.path.normpath(root))
        for path in sorted(glob.glob(os.path.join(root, "models", "*", "*", "*"))):
            if not os.path.isfile(path) or not path.lower().endswith(MODEL_SUFFIXES):
                continue
            parts = os.path.normpath(path).split(os.sep)
            method, entry, fname = parts[-3], parts[-2].upper(), parts[-1]
            kind = model_kind(method, fname)
            key = (method, entry, kind, fname.lower())
            if key in found:
                warnings.append("%s %s %s: in %s and in %s (the last one is used)" % (method, entry, fname, found[key]["source"], label))
            found[key] = {"method": method, "entry": entry, "model_kind": kind, "primary": kind in PRIMARY_KINDS, "model_file": fname, "path": path, "source": label}
    return sorted(found.values(), key=lambda m: (m["entry"], m["method"], m["model_kind"], m["model_file"])), warnings


def prepare_natives(entries, cif_dir, out_dir):
    """Reads <cif_dir>/<ENTRY>.cif for every entry row (dict with pdb_id and modeled_residues), writes <out_dir>/<ENTRY>_native_ca.pdb and returns
    {entry: (Trace, path, note)}. The note says when the residues differ from `modeled_residues` of the entries table."""
    os.makedirs(out_dir, exist_ok=True)
    out = {}
    for row in entries:
        entry = row["pdb_id"].upper()
        src = None
        for cand in (os.path.join(cif_dir, entry + ".cif"), os.path.join(cif_dir, entry.lower() + ".cif"), os.path.join(cif_dir, entry + ".pdb")):
            if os.path.exists(cand):
                src = cand
                break
        if src is None:
            raise FileNotFoundError("no deposited model for %s in %s" % (entry, cif_dir))
        trace = read_trace(src)
        path = write_pdb(trace, os.path.join(out_dir, entry + "_native_ca.pdb"))
        note = ""
        want = row.get("modeled_residues")
        if want not in (None, "") and int(float(want)) != trace.n_residues:
            note = "%d amino-acid residues with a CA atom, entries table says %s" % (trace.n_residues, want)
        out[entry] = (trace, path, note)
    return out


def write_scores(rows, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=SCORE_COLUMNS, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)


COORDINATE_FILES = ("query_superposed.pdb", "query_frame_fixed.pdb", "pred_merged.pdb", "native_merged.pdb", "native_ca.pdb")


def score_all(models, natives, usalign, phenix_env, out_dir, workers=4, log=print, only_primary=False, keep_coordinates=False, progress=None):
    """Scores every model of collect_models; returns the rows (and writes scores_long.csv and the raw outputs under out_dir/raw).
    The coordinate files that the scoring writes (traces of the predictions: for MICA they derive from the private AlphaFold predictions) are removed after each model
    unless keep_coordinates is set; progress(rows so far) is called after every model."""
    import threading
    jobs = [m for m in models if (m["primary"] or not only_primary)]
    os.makedirs(out_dir, exist_ok=True)
    rows = [None] * len(jobs)
    lock = threading.Lock()

    def work(i):
        m = jobs[i]
        wd = os.path.join(out_dir, "raw", m["entry"], os.path.splitext(m["model_file"])[0].replace("(", "_").replace(")", "_").replace("+", "_"))
        if m["entry"] not in natives:
            r = {k: "" for k in SCORE_COLUMNS}
            r.update(status="no_native", note="no deposited model for this entry")
            trace, native_file, native_note = None, None, ""
        else:
            trace, native_file, native_note = natives[m["entry"]]
            try:
                r = score_model(usalign, phenix_env, trace, m["path"], wd, native_file=native_file)
            except Exception as e:                    # a bug must not cost the other models
                r = {k: "" for k in SCORE_COLUMNS}
                r.update(status="no_alignment", note="unexpected error: %r" % (e,), native_residues=trace.n_residues)
            if not keep_coordinates:
                for name in COORDINATE_FILES:
                    try:
                        os.remove(os.path.join(wd, name))
                    except OSError:
                        pass
        r.update(entry=m["entry"], method=m["method"], model_kind=m["model_kind"], primary=m["primary"], model_file=m["model_file"], source=m["source"])
        if native_note:
            r["note"] = ("native: " + native_note + (" | " + r["note"] if r["note"] else ""))[:600]
        with lock:
            rows[i] = r
            log("[%d/%d] %-5s %-12s %-22s %-10s TM %-7s fNS %-7s CAmatch %-7s %ss" % (i + 1, len(jobs), m["entry"], m["method"], m["model_kind"], r["status"], r["tm_score"], r["tm_score_fns"], r["ca_match"], r["seconds"]))
            if progress:
                progress([x for x in rows if x is not None])
        return r

    with ThreadPoolExecutor(max_workers=max(1, workers)) as pool:
        list(pool.map(work, range(len(jobs))))
    write_scores(rows, os.path.join(out_dir, "scores_long.csv"))
    return rows


def read_entries(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    one = sub.add_parser("one", help="score one model")
    one.add_argument("--usalign", required=True)
    one.add_argument("--native", required=True, help="deposited model (pdb or mmCIF)")
    one.add_argument("--pred", required=True)
    one.add_argument("--phenix-env", help="a shell file that makes the phenix.* programs available (not given: Phenix is not run)")
    one.add_argument("--workdir", required=True)
    bat = sub.add_parser("batch", help="score all the models of result folders")
    bat.add_argument("--usalign", required=True)
    bat.add_argument("--roots", nargs="+", required=True, help="result folders (with models/<method>/<ENTRY>/), oldest first")
    bat.add_argument("--entries", required=True, help="data/entries_2026.csv")
    bat.add_argument("--natives", required=True, help="folder with <ENTRY>.cif of the deposited models")
    bat.add_argument("--out", required=True)
    bat.add_argument("--phenix-env")
    bat.add_argument("--workers", type=int, default=4)
    bat.add_argument("--only-primary", action="store_true")
    args = p.parse_args(argv)
    if args.cmd == "one":
        native = read_trace(args.native)
        res = score_model(args.usalign, args.phenix_env, native, args.pred, args.workdir)
        print(json.dumps(res, indent=1))
        return 0 if res["status"] == "ok" else 2
    entries = read_entries(args.entries)
    natives = prepare_natives(entries, args.natives, os.path.join(args.out, "natives"))
    models, warnings = collect_models(args.roots)
    for w in warnings:
        print("WARNING:", w)
    rows = score_all(models, natives, args.usalign, args.phenix_env, args.out, args.workers, only_primary=args.only_primary)
    return 0 if all(r["status"] == "ok" for r in rows) else 2


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile data/entries_2026.csv
emdb_id,pdb_id,emdb_release_date,pdb_release_date,resolution_A,stratum,modeled_residues,unique_chain_seqs,unk_fraction,cc_mask,cluster_id,seed_rank,max_id_to_pre2026_pdb,cluster_size,longest_seq_len,n_chains,map_size_mb,cif_url,map_url
EMD-55586,9T5K,2026-04-22,2026-04-22,2.6,le3.0,381,1,0.0,0.8200,9T5K,1,0.614,1,453,1,250.0,https://files.rcsb.org/download/9T5K.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55586/map/emd_55586.map.gz
EMD-68251,22FX,2026-03-04,2026-03-04,1.24,le3.0,173,1,0.0,0.7388,22FX,2,1.000,1,182,1,75.2,https://files.rcsb.org/download/22FX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68251/map/emd_68251.map.gz
EMD-61564,9JKK,2026-04-08,2026-04-08,3.0,le3.0,530,1,0.0,0.8420,9W97,3,1.000,39,620,1,65.5,https://files.rcsb.org/download/9JKK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61564/map/emd_61564.map.gz
EMD-56022,9TKA,2026-07-29,2026-07-29,2.7,le3.0,247,1,0.0,0.7625,9THK,4,0.000,3,293,1,182.3,https://files.rcsb.org/download/9TKA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56022/map/emd_56022.map.gz
EMD-65930,9WFL,2026-04-01,2026-04-01,2.75,le3.0,682,2,0.0,0.8396,9WF3,5,1.000,6,697,2,65.5,https://files.rcsb.org/download/9WFL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65930/map/emd_65930.map.gz
EMD-65077,9VI4,2026-03-11,2026-03-11,1.8,le3.0,516,1,0.0,0.8749,9P4L,6,0.984,9,516,1,1024.0,https://files.rcsb.org/download/9VI4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65077/map/emd_65077.map.gz
EMD-74403,9ZLH,2026-09-16,2026-09-16,2.14,le3.0,206,1,0.0,0.8468,9ZK8,7,1.000,7,206,1,65.5,https://files.rcsb.org/download/9ZLH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74403/map/emd_74403.map.gz
EMD-65363,9VUH,2026-07-15,2026-07-15,2.7,le3.0,634,3,0.0,0.7439,9VUH,8,1.000,1,365,3,128.0,https://files.rcsb.org/download/9VUH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65363/map/emd_65363.map.gz
EMD-70305,9OC0,2026-05-06,2026-05-06,2.97,le3.0,512,2,0.0,0.8262,9OBT,11,1.000,8,862,4,332.8,https://files.rcsb.org/download/9OC0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70305/map/emd_70305.map.gz
EMD-76936,13BD,2026-06-17,2026-06-17,2.65,le3.0,461,1,0.0,0.8841,9OVK,12,0.954,25,1271,2,281.2,https://files.rcsb.org/download/13BD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76936/map/emd_76936.map.gz
EMD-66160,9WQ6,2026-09-16,2026-09-16,3.4,gt3.0_le4.0,261,1,0.0,0.7629,9W3F,4,1.000,11,339,1,128.0,https://files.rcsb.org/download/9WQ6.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66160/map/emd_66160.map.gz
EMD-64777,9V4D,2026-07-01,2026-07-01,3.07,gt3.0_le4.0,276,2,0.0,0.8202,25IF,5,0.503,6,393,2,54.0,https://files.rcsb.org/download/9V4D.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64777/map/emd_64777.map.gz
EMD-70828,9OTA,2026-05-27,2026-05-27,3.2,gt3.0_le4.0,534,1,0.0,0.8452,9E94,7,1.000,2,270,2,68.7,https://files.rcsb.org/download/9OTA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70828/map/emd_70828.map.gz
EMD-62507,9KQM,2026-06-03,2026-06-03,3.5,gt3.0_le4.0,403,1,0.0,0.7874,9KKN,8,0.997,9,637,1,65.5,https://files.rcsb.org/download/9KQM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62507/map/emd_62507.map.gz
EMD-66088,9WME,2026-09-23,2026-09-23,3.2,gt3.0_le4.0,446,2,0.0,0.7050,9WME,9,1.000,1,657,2,22.8,https://files.rcsb.org/download/9WME.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66088/map/emd_66088.map.gz
EMD-64920,9VBD,2026-04-08,2026-04-08,3.26,gt3.0_le4.0,521,1,0.0,0.7685,9VBD,15,0.000,2,893,1,128.0,https://files.rcsb.org/download/9VBD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64920/map/emd_64920.map.gz
EMD-54346,9RWS,2026-04-29,2026-04-29,3.28,gt3.0_le4.0,512,1,0.0,0.8467,9RWS,16,0.441,1,551,1,128.0,https://files.rcsb.org/download/9RWS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54346/map/emd_54346.map.gz
EMD-66226,9WTX,2026-09-23,2026-09-23,3.37,gt3.0_le4.0,500,1,0.0,0.7238,9US8,17,0.763,3,551,1,128.0,https://files.rcsb.org/download/9WTX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66226/map/emd_66226.map.gz
EMD-73762,9Z24,2026-06-10,2026-06-10,3.1,gt3.0_le4.0,338,1,0.0,0.8749,9Z1X,18,1.000,4,1214,1,221.2,https://files.rcsb.org/download/9Z24.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73762/map/emd_73762.map.gz
EMD-80558,26DK,2026-08-26,2026-08-26,3.5,gt3.0_le4.0,191,1,0.0,0.7407,26DK,19,0.000,2,197,1,16.0,https://files.rcsb.org/download/26DK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80558/map/emd_80558.map.gz


In [ ]:
EXPECTED = {'scripts/nbtools.py': '9bda95fce7f7fbe0c373e315740f555fc27942a08971ce2fe1b5cae6af6d748d', 'scripts/score_models.py': '57b40ab95807b216edaf83741806523772a1a6febc48d29877ae3f72426cbf90', 'data/entries_2026.csv': 'cb695bb04ef09bf48f8c2848f91a2580c0b3c15f2a8f1e580659d86f0097a5da'}
bad = [p for p, h in EXPECTED.items() if not os.path.exists(p) or sha256_file(p) != h]
if bad:
    fail("these files were not written as expected: %s. Use Run all again; if it happens again, send me this message." % ", ".join(bad))
else:
    log("all %d files written and checked (hashes match)" % len(EXPECTED))


## Step 2b. Progress lines, a zip that is refreshed, and the last cell that works after a restart

This cell adds three things that the first pilot lacked: a progress line every few minutes while a program is silent, the file `state.json`, and a zip
(`scoring_cryozeta_d_partial.zip` in the output files) that is written again after every stage and every ten minutes, so a stop in the middle never leaves nothing.

In [ ]:
sys.path.insert(0, os.path.join(WORK, "scripts"))
import nbtools

# Kaggle's kernel sets MPLBACKEND=module://matplotlib_inline.backend_inline. Every program started from this notebook inherits it, and `import matplotlib.pyplot`
# then fails in any environment without matplotlib-inline (ModelAngelo's, in the first pilot: the run stopped after 5 minutes). Agg needs nothing.
os.environ["MPLBACKEND"] = "Agg"

NOTEBOOK = "scoring_cryozeta_d"
STATE_FILE = os.path.join(WORK, "state.json")
PARTIAL_ZIP = nbtools.partial_zip_path(OUT_DIR, NOTEBOOK)
PACK_LOCK = threading.Lock()
LAST_PARTIAL = {"t": 0.0}


def save_state():
    try:
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(STATE, f, indent=1, default=str)
    except Exception as ex:
        log("could not save state.json: %s" % ex)


def pack_now(min_gap=0):
    """Writes the partial zip again (same name every time). min_gap: do nothing if the last one is younger than this many seconds."""
    with PACK_LOCK:
        if time.time() - LAST_PARTIAL["t"] < min_gap:
            return
        LAST_PARTIAL["t"] = time.time()
        try:
            save_state()
            nbtools.pack(WORK, PARTIAL_ZIP, NOTEBOOK + "_partial", secrets=SECRETS)
        except Exception as ex:
            log("the partial zip could not be written: %s" % ex)


BEAT = float(os.environ.get("MICA_BEAT", "120"))          # seconds between progress lines (the tests make it short)


def run_watched(cmd, label, watch=(), beat=None, zip_every=600, **kw):
    """run() with a progress line every `beat` seconds (time, load, free RAM, busiest processes, last line of the files in `watch`)."""
    hb = nbtools.Heartbeat(label, log, watch=watch, every=beat or BEAT, on_tick=lambda: pack_now(zip_every))
    hb.start()
    try:
        return run(cmd, **kw)
    finally:
        hb.stop()


def finish(complete, extra_info=None):
    """Writes run_info.json and the final zip; returns the path of the zip."""
    info = {"notebook": NOTEBOOK, "started_utc": STATE["started"], "finished_utc": nbtools.utc_now().isoformat(), "elapsed_minutes": round((time.time() - STATE["t0"]) / 60),
            "problem": STATE["problem"], "complete": complete, "platform": PLATFORM, "python": platform.python_version(), "os": platform.platform(), "gpu": STATE["gpu"],
            "cpu_cores": os.cpu_count(), "ram_gb": meminfo_gb(), "state": STATE}
    info.update(extra_info or {})
    with open(os.path.join(WORK, "run_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, default=str)
    save_state()
    path, n, left_out = nbtools.final_pack(WORK, OUT_DIR, NOTEBOOK, complete, secrets=SECRETS)
    log("zip written: %s (%d files, %d MB)" % (path, n, os.path.getsize(path) // 1000000))
    if left_out:
        log("left out of the zip: %s" % "; ".join(left_out))
    return path


def cpu_facts():
    """How many processors the program sees and how many it may really use (a container can show more cores than its limit allows)."""
    import multiprocessing
    facts = {"os.cpu_count": os.cpu_count(), "multiprocessing.cpu_count": multiprocessing.cpu_count(),
             "usable (sched_getaffinity)": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None, "nproc": sh("nproc 2>&1"),
             "cgroup v2 cpu.max": sh("cat /sys/fs/cgroup/cpu.max 2>&1"), "cgroup v1 quota": sh("cat /sys/fs/cgroup/cpu/cpu.cfs_quota_us 2>&1"),
             "MICA's nproc formula min(max(int(0.6 * cpu_count), 1), 48)": min(max(int(0.6 * multiprocessing.cpu_count()), 1), 48)}
    with open(os.path.join(WORK, "cpu_facts.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join("%s: %s" % kv for kv in facts.items()) + "\n")
    log("processors: " + " | ".join("%s %s" % (k.split(" ")[0], v) for k, v in facts.items()))
    STATE["cpu_facts"] = facts


save_state()
cpu_facts()
log("progress lines, state.json and the partial zip (%s) are ready" % PARTIAL_ZIP)


## Step 3. Install Phenix

Needed once per session (about 15 to 25 minutes). If the user name and password are not found in the Secrets, two boxes ask for them.
If Phenix is already installed in this session, this step is skipped.

In [ ]:
STATE['ok_before_phenix'] = bool(STATE['ok'])


In [ ]:
PHENIX_URL = "https://phenix-online.org/download/phenix/release/send_octet_stream.cgi?version=2.2.1-6174&file=Phenix-2.2.1-6174-Linux-x86_64.sh"
PHENIX_VERSION = "2.2.1-6174"
INSTALLER = os.path.join(BIG, "Phenix-2.2.1-6174-Linux-x86_64.sh")
PREFIX = os.path.join(BIG, "phenix-" + PHENIX_VERSION)
MIN_INSTALLER_MB = 50
MIN_FREE_GB = 20


def find_phenix_env():
    for pattern in ("phenix_env.sh", "*/phenix_env.sh", "*/*/phenix_env.sh"):
        hits = sorted(glob.glob(os.path.join(PREFIX, pattern)))
        if hits:
            return hits[0]
    return None


def phenix_works(env_script):
    r = subprocess.run(["bash", "-c", 'source "$0" && phenix.version', env_script], capture_output=True, text=True, errors="replace")
    lines = [ln.strip() for ln in (r.stdout + r.stderr).splitlines() if ln.strip()]
    text = " | ".join(lines[:3])[:200]
    if r.returncode == 0:
        return True, text
    r2 = subprocess.run(["bash", "-c", 'source "$0" && command -v phenix.real_space_refine', env_script], capture_output=True, text=True, errors="replace")
    if r2.returncode == 0:
        return True, "phenix.real_space_refine found at %s (phenix.version said: %s)" % (r2.stdout.strip(), text or "nothing")
    return False, text or "phenix.version did not run"


def download_installer():
    import requests
    if os.path.exists(INSTALLER) and os.path.getsize(INSTALLER) >= MIN_INSTALLER_MB * 1e6:
        log("%s is already here (%d MB)" % (INSTALLER, os.path.getsize(INSTALLER) // 1000000))
        return True
    user = secret("PHENIX_USER", "Phenix user name (from the email), then Enter: ", hidden=False)
    password = secret("PHENIX_PASSWORD", "Phenix password (from the email), then Enter: ")
    if not user or not password:
        fail("no Phenix user name or password found. Kaggle: Add-ons -> Secrets, add PHENIX_USER and PHENIX_PASSWORD and tick both for this notebook. "
             "Colab: the key icon on the left, add the same two secrets and switch on notebook access. Then Run all again.")
        return False
    if len(password) >= 6:
        SECRETS.append(password)
    last = ""
    for attempt in range(1, 4):
        try:
            with requests.get(PHENIX_URL, auth=(user, password), stream=True, timeout=(30, 120)) as r:
                if r.status_code == 401:
                    fail("the Phenix server did not accept the user name and password (HTTP 401). The password changes on the 1st of every "
                         "month, so use the one from the newest email, and type both exactly as they appear. Then use Run all again.")
                    return False
                if r.status_code != 200:
                    last = "HTTP %d" % r.status_code
                    raise IOError(last)
                total = int(r.headers.get("Content-Length") or 0)
                done, shown = 0, -1
                with open(INSTALLER + ".part", "wb") as f:
                    for block in r.iter_content(1 << 22):
                        f.write(block)
                        done += len(block)
                        mark = done * 10 // total if total else done // 250000000
                        if mark != shown:
                            shown = mark
                            log("  downloaded %d MB%s" % (done // 1000000, " of %d MB" % (total // 1000000) if total else ""))
            if total and done != total:
                last = "the file is incomplete (%d of %d bytes)" % (done, total)
                raise IOError(last)
            with open(INSTALLER + ".part", "rb") as f:
                head = f.read(2)
            if head != b"#!" or done < MIN_INSTALLER_MB * 1e6:
                fail("the Phenix server sent %d bytes that do not look like the installer (it is probably a web page, not the file). Check the "
                     "user name and password, then use Run all again." % done)
                os.remove(INSTALLER + ".part")
                return False
            os.replace(INSTALLER + ".part", INSTALLER)
            return True
        except (requests.RequestException, IOError) as ex:
            last = last or str(ex)[:150]
            log("download attempt %d of 3 failed: %s" % (attempt, last))
            time.sleep(10 * attempt)
            last = ""
    fail("the Phenix installer could not be downloaded after 3 attempts. Use Run all again in a few minutes.")
    return False


def install_phenix():
    env = find_phenix_env()
    if env:
        ok, text = phenix_works(env)
        if ok:
            log("Phenix is already installed: %s (%s)" % (text, env))
            STATE.update(phenix_env=env, phenix_version=text)
            return
    free = shutil.disk_usage(BIG).free / 1e9
    if free < MIN_FREE_GB:
        fail("only %d GB of disk are free in %s; Phenix needs about %d GB or more while it installs. Kaggle: use a new session (Session options -> restart); "
             "Colab: Runtime -> Disconnect and delete runtime. Then Run all again." % (free, BIG, MIN_FREE_GB))
        return
    if not download_installer():
        return
    STATE["installer_sha256"] = sha256_file(INSTALLER)
    log("installer sha256: %s" % STATE["installer_sha256"])
    tmp = os.path.join(BIG, "tmp_phenix_install")
    os.makedirs(tmp, exist_ok=True)
    shutil.rmtree(PREFIX, ignore_errors=True)
    log("installing Phenix (15 to 25 minutes; a line is printed now and then) ...")
    t0 = time.time()
    rc, last, _ = run(["bash", INSTALLER, "-b", "-p", PREFIX], env={"TMPDIR": tmp}, every=60)
    shutil.rmtree(tmp, ignore_errors=True)
    if rc != 0:
        log("\n".join(last[-25:]))
        fail("the Phenix installer stopped with exit code %d (the last lines are above). If they say 'No space left', start a fresh session and "
             "run again; otherwise send me the zip." % rc)
        return
    env = find_phenix_env()
    ok, text = phenix_works(env) if env else (False, "phenix_env.sh was not found under %s" % PREFIX)
    if not ok:
        fail("Phenix was installed but does not start: %s" % text)
        return
    STATE.update(phenix_env=env, phenix_version=text)
    log("Phenix installed in %d minutes: %s" % ((time.time() - t0) // 60, text))
    os.remove(INSTALLER)


def phenix_probe():
    """Which Phenix programs exist, and what chain_comparison says about itself (its exact usage is part of the protocol)."""
    script = ('source "$0" && for t in phenix.dock_in_map phenix.real_space_refine phenix.chain_comparison phenix.version; do '
              'printf "%s: " $t; command -v $t || echo MISSING; done; echo; phenix.version 2>&1 | head -5; echo; echo "==== phenix.chain_comparison --help"; '
              'phenix.chain_comparison --help 2>&1 | head -120')
    try:
        r = subprocess.run(["bash", "-c", script, STATE["phenix_env"]], capture_output=True, text=True, errors="replace", timeout=600)
        text = r.stdout + r.stderr
    except subprocess.TimeoutExpired:
        text = "the probe of the Phenix programs did not finish within 10 minutes\n"
    with open("phenix_help.txt", "w", encoding="utf-8") as f:
        f.write(text)
    log("\n".join(text.splitlines()[:8]))
    missing = [t for t in ("phenix.dock_in_map", "phenix.real_space_refine", "phenix.chain_comparison") if (t + ": MISSING") in text]
    if missing:
        log("WARNING: this Phenix has no %s (see phenix_help.txt). MICA needs phenix.dock_in_map and phenix.real_space_refine; the scoring needs phenix.chain_comparison." % ", ".join(missing))


if STATE["ok"]:
    install_phenix()
    if STATE["ok"]:
        phenix_probe()
else:
    log("skipped: an earlier step stopped")


In [ ]:
# Phenix is needed for the columns of chain_comparison only. If it could not be installed, the notebook goes on without it (US-align and the own counts are complete)
# and says so at the end; the zip is then marked INCOMPLETE.
if STATE.get("ok_before_phenix") and not STATE["ok"]:
    STATE["phenix_problem"] = STATE["problem"]
    STATE["ok"], STATE["problem"], STATE["phenix_env"] = True, "", None
    log("\n" + "!" * 72 + "\nPhenix could not be installed (%s).\nThe notebook goes on without it: the Phenix columns stay empty; US-align and the own counts are computed.\n" % STATE["phenix_problem"][:300] + "!" * 72)


## Step 4. Python packages and US-align

`gemmi` (reads the models), `scipy` and `numpy` come from PyPI; US-align is compiled from the pinned commit (the same one as in every other notebook).

In [ ]:
USALIGN_GIVEN = None                      # None: compile it; a path is only used by the tests
USALIGN_REPO = "https://github.com/pylelab/USalign"
USALIGN_COMMIT = "746ad8a2a4dea9f317075cc482df0ecb5440bdea"
def git_checkout(repo, commit, folder):
    if os.path.isdir(os.path.join(folder, ".git")) and sh("git -C %s rev-parse HEAD" % folder) == commit:
        return True
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder)
    for cmd in (["git", "init", "-q", folder], ["git", "-C", folder, "remote", "add", "origin", repo],
                ["git", "-C", folder, "fetch", "-q", "--depth", "1", "origin", commit], ["git", "-C", folder, "checkout", "-q", "FETCH_HEAD"]):
        rc, last, _ = run(cmd)
        if rc != 0:
            log("\n".join(last))
            return False
    return sh("git -C %s rev-parse HEAD" % folder) == commit


def build_usalign():
    if USALIGN_GIVEN:
        STATE["usalign"] = USALIGN_GIVEN
    else:
        src = os.path.join(BIG, "USalign_src")
        exe = os.path.join(src, "USalign")
        if not os.path.exists(exe):
            if not git_checkout(USALIGN_REPO, USALIGN_COMMIT, src):
                fail("US-align could not be fetched at commit %s" % USALIGN_COMMIT)
                return False
            rc, last, _ = run(["g++", "-O3", "-ffast-math", "-o", exe, os.path.join(src, "USalign.cpp"), "-lm"], cwd=src, every=10)
            if rc != 0:
                fail("US-align could not be compiled (see the lines above). Send me the zip.")
                return False
        STATE["usalign"] = exe
    r = subprocess.run([STATE["usalign"], "-h"], capture_output=True, text=True, errors="replace")
    head = [ln for ln in (r.stdout + r.stderr).splitlines() if "Version" in ln]
    STATE["usalign_version"] = head[0].strip() if head else "unknown"
    log("US-align: %s (%s)" % (STATE["usalign"], STATE["usalign_version"]))
    return True



def install_packages():
    missing = []
    for module, package in (("numpy", "numpy"), ("scipy", "scipy"), ("gemmi", "gemmi")):
        try:
            __import__(module)
        except ImportError:
            missing.append(package)
    if missing:
        rc, last, _ = run([sys.executable, "-m", "pip", "install", "-q"] + missing, every=10)
        if rc != 0:
            fail("pip could not install %s (see the lines above). Send me the zip." % ", ".join(missing))
            return False
    import gemmi
    log("gemmi %s, numpy %s, scipy %s" % (gemmi.__version__, __import__("numpy").__version__, __import__("scipy").__version__))
    STATE["gemmi"] = gemmi.__version__
    return True


if STATE["ok"]:
    if install_packages():
        build_usalign()
else:
    log("skipped: an earlier step stopped")


## Step 5. The models of the other notebooks

Looks in `/kaggle/input` for the result zips (`*_result_*.zip`) and for result folders (a folder with `models/<method>/<ENTRY>/`), oldest first, takes the models and the status tables out of them
(never the private AlphaFold-derived state), and lists what it found. A model that is in two sources is taken from the later one and the notebook says so.

In [ ]:
import glob, zipfile
sys.path.insert(0, os.path.join(WORK, "scripts"))
import score_models

INPUT_ROOTS = [r for r in (os.environ.get("SCORING_INPUT"), "/kaggle/input") if r and os.path.isdir(r)]
INPUTS_DIR = os.path.join(WORK, "downloads", "inputs")
STATUS_DIR = os.path.join(WORK, "status")
WANTED_FILES = ("method_status.csv", "mica_status.csv", "pending.csv", "run_info.json", "metrics_usalign.csv", "docking_domains.csv")
PRUNE = {"mica_state", "AF3_results", "af3_upload", "grids", "temp_maps", "models", "__pycache__"}
MODELS = []
SOURCES = []


def stamp_of(path):
    m = re.search(r"(\d{8}-\d{4})", os.path.basename(path.rstrip(os.sep)))
    return m.group(1) if m else ""


def find_sources(max_depth=6):
    zips, folders = [], []
    for root in INPUT_ROOTS:
        base = root.rstrip(os.sep).count(os.sep)
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count(os.sep) - base >= max_depth:
                dirnames[:] = []
            if os.path.isdir(os.path.join(dirpath, "models")) and glob.glob(os.path.join(dirpath, "models", "*", "*")):
                folders.append(dirpath)
            dirnames[:] = [d for d in dirnames if d not in PRUNE]
            for n in filenames:
                if n.endswith(".zip") and "_result_" in n and not n.endswith("_partial.zip"):
                    zips.append(os.path.join(dirpath, n))
    return sorted(zips, key=lambda p: (stamp_of(p), p)), sorted(folders, key=lambda p: (stamp_of(p), p))


def extract_zip(path):
    label = os.path.splitext(os.path.basename(path))[0]
    dest = os.path.join(INPUTS_DIR, label)
    shutil.rmtree(dest, ignore_errors=True)
    n_models = 0
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if name.endswith("/"):
                continue
            parts = name.split("/")
            rel = name if parts[0] == "models" else "/".join(parts[1:])
            if rel.startswith("models/") or rel in WANTED_FILES:
                target = os.path.join(dest, rel)
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with z.open(name) as src, open(target, "wb") as out:
                    shutil.copyfileobj(src, out)
                n_models += rel.startswith("models/")
    return dest, n_models


def gather():
    zips, folders = find_sources()
    log("input folders: %s" % (", ".join(INPUT_ROOTS) or "none"))
    log("result zips found: %d, result folders found: %d" % (len(zips), len(folders)))
    roots = []
    for z in zips:
        try:
            dest, n = extract_zip(z)
        except (zipfile.BadZipFile, OSError) as ex:
            log("  %s: cannot be read (%s)" % (z, ex))
            continue
        with open(z, "rb") as f:
            digest = hashlib.sha256(f.read()).hexdigest()
        log("  zip    %s: %d model files" % (os.path.relpath(z, "/") if z.startswith("/") else z, n))
        roots.append(dest)
        SOURCES.append({"label": os.path.basename(dest), "kind": "zip", "path": z, "bytes": os.path.getsize(z), "sha256": digest, "model_files": n})
    for d in folders:
        if any(os.path.abspath(d).startswith(os.path.abspath(r) + os.sep) or os.path.abspath(d) == os.path.abspath(r) for r in roots):
            continue
        n = len(glob.glob(os.path.join(d, "models", "*", "*", "*")))
        log("  folder %s: %d model files" % (d, n))
        roots.append(d)
        SOURCES.append({"label": os.path.basename(d.rstrip(os.sep)), "kind": "folder", "path": d, "bytes": "", "sha256": "", "model_files": n})
    return roots


def copy_status(roots):
    os.makedirs(STATUS_DIR, exist_ok=True)
    for root in roots:
        label = os.path.basename(root.rstrip(os.sep))
        for name in WANTED_FILES:
            src = os.path.join(root, name)
            if os.path.isfile(src):
                shutil.copy2(src, os.path.join(STATUS_DIR, "%s__%s" % (label, name)))


def table_of_models():
    counts = {}
    for m in MODELS:
        counts.setdefault((m["method"], m["model_kind"]), set()).add(m["entry"])
    log("\n%-14s %-22s %7s" % ("method", "kind", "entries"))
    for (method, kind), entries in sorted(counts.items()):
        log("%-14s %-22s %7d" % (method, kind, len(entries)))


if STATE["ok"]:
    roots = gather()
    if not roots:
        fail("no result zip and no result folder was found in the input folders. Add the outputs of the other notebooks as inputs of this notebook "
             "(Add Input -> Notebook Output Files -> gpu_cryozeta_d), then run again.")
    else:
        MODELS, problems = score_models.collect_models(roots)
        for w in problems:
            log("WARNING: " + w)
        if not MODELS:
            fail("the inputs hold no model (no models/<method>/<ENTRY>/ folder with .pdb or .cif files)")
        else:
            copy_status(roots)
            table_of_models()
            STATE["sources"] = SOURCES
            log("\n%d model files of %d entries; %d of them are the primary file of their method" % (len(MODELS), len({m["entry"] for m in MODELS}), sum(m["primary"] for m in MODELS)))
            pack_now()
else:
    log("skipped: an earlier step stopped")


## Step 6. The deposited models, and a test of the scoring

Downloads the deposited model of every entry that has models (the `cif_url` of `data/entries_2026.csv`), keeps its amino-acid residues (C-alpha trace, first model, first alternative location) and
checks the residue count against the entries table. Then the deposited model of the first entry is scored against itself: US-align must give TM-score 1 and every residue aligned, Phenix must
find every residue. If Phenix does not pass this test, the notebook goes on without it (the own counts are still written) and says so in the first line of the result.

In [ ]:
import urllib.request

NATIVE_DIR = os.path.join(WORK, "native")
SCORING_DIR = os.path.join(WORK, "scoring")
NATIVES = {}


def download_to(url, dest, tries=4, timeout=120):
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r, open(dest + ".part", "wb") as f:
                shutil.copyfileobj(r, f)
            os.replace(dest + ".part", dest)
            return True
        except Exception as ex:
            log("  download of %s failed (try %d of %d): %s" % (url, i + 1, tries, ex))
            time.sleep(5 * (i + 1))
    return False


def get_natives():
    rows = {r["pdb_id"].upper(): r for r in score_models.read_entries(os.path.join(WORK, "data", "entries_2026.csv"))}
    os.makedirs(NATIVE_DIR, exist_ok=True)
    wanted = sorted({m["entry"] for m in MODELS})
    have = []
    for entry in wanted:
        if entry not in rows:
            log("  %s is not in data/entries_2026.csv: its models are not scored" % entry)
            continue
        dest = os.path.join(NATIVE_DIR, entry + ".cif")
        if os.path.exists(dest) or download_to(rows[entry]["cif_url"], dest):
            have.append(rows[entry])
        else:
            log("  the deposited model of %s could not be downloaded: its models are not scored (status no_native)" % entry)
    got = score_models.prepare_natives(have, NATIVE_DIR, os.path.join(NATIVE_DIR, "ca")) if have else {}
    out = []
    for entry, (trace, path, note) in sorted(got.items()):
        out.append({"entry": entry, "residues": trace.n_residues, "chains": trace.n_chains, "note": note, "sha256": sha256_file(os.path.join(NATIVE_DIR, entry + ".cif"))})
        if note:
            log("  %s: %s" % (entry, note))
    write_csv(os.path.join(WORK, "scoring", "natives.csv"), ["entry", "residues", "chains", "note", "sha256"], out)
    return got


def write_csv(path, columns, rows):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)


def selftest(natives):
    """The first deposited model against itself."""
    entry = sorted(natives)[0]
    trace, path, _ = natives[entry]
    r = score_models.score_model(STATE["usalign"], STATE["phenix_env"], trace, path, os.path.join(SCORING_DIR, "selftest"), native_file=path)
    problems = []
    if r["status"] != "ok" or not (isinstance(r["tm_score"], float) and r["tm_score"] > 0.999) or r["aligned_ca_length"] != trace.n_residues:
        problems.append("US-align: %s" % {k: r[k] for k in ("status", "tm_score", "aligned_ca_length", "note")})
    phenix_bad = None
    if STATE["phenix_env"]:
        if r["cc_status"] != "ok" or r["cc_status_ff"] != "ok" or r["ca_match"] != 100.0 or r["ca_found"] != 100.0 or r["seq_match"] != 100.0:
            phenix_bad = "Phenix: %s" % {k: r[k] for k in ("cc_status", "cc_status_ff", "ca_match", "ca_found", "seq_match", "note")}
    log("self-test on %s (%d residues): TM-score %s, fNS %s, own match %s, Phenix match %s found %s sequence %s" % (
        entry, trace.n_residues, r["tm_score"], r["tm_score_fns"], r["own_ca_match"], r["ca_match"], r["ca_found"], r["seq_match"]))
    for name in score_models.COORDINATE_FILES:
        try:
            os.remove(os.path.join(SCORING_DIR, "selftest", name))
        except OSError:
            pass
    return problems, phenix_bad


if STATE["ok"] and MODELS:
    NATIVES = get_natives()
    if not NATIVES:
        fail("no deposited model could be read: nothing can be scored")
    else:
        problems, phenix_bad = selftest(NATIVES)
        if problems:
            fail("the self-test of the scoring failed: " + " | ".join(problems))
        STATE["phenix_used"] = bool(STATE["phenix_env"])
        if phenix_bad:
            log("\n" + "!" * 72 + "\nPhenix chain_comparison did not pass the self-test, so the Phenix columns are NOT computed.\n" + phenix_bad + "\n" + "!" * 72)
            STATE["phenix_env_unused"] = STATE["phenix_env"]
            STATE["phenix_used"] = False
        elif not STATE["phenix_env"]:
            log("\n" + "!" * 72 + "\nPhenix is not available: the Phenix columns are NOT computed (the own counts are).\n" + "!" * 72)
else:
    log("skipped: an earlier step stopped")


## Step 7. Score every model

`WORKERS` models at a time: US-align `-mm 1` (the original file), US-align `-mm 5` (one chain per structure), Phenix chain_comparison twice, the own counts. One line per model; the raw outputs of the programs
are kept (the coordinate files that the scoring writes are removed after each model). A model that cannot be read is `no_model`, a model whose alignment fails or runs out of time (15 minutes) is
`no_alignment`; both stay in the table.

In [ ]:
SCORE_ROWS = []


def progress(rows):
    if len(rows) % 10 == 0:
        score_models.write_scores(rows, os.path.join(SCORING_DIR, "scores_long.csv"))
        pack_now(600)


if STATE["ok"] and MODELS and NATIVES:
    t_score = time.time()
    SCORE_ROWS = score_models.score_all(MODELS, NATIVES, STATE["usalign"], STATE["phenix_env"] if STATE.get("phenix_used") else None, SCORING_DIR, workers=WORKERS, log=log, progress=progress)
    log("\n%d models scored in %d minutes" % (len(SCORE_ROWS), round((time.time() - t_score) / 60)))
    pack_now()
else:
    log("skipped: an earlier step stopped")


## Step 8. The tables

`scoring/scores_long.csv` (every model; the columns are explained in `scripts/score_models.py`), `scoring/scores_primary.csv` (the primary file of each method for each entry), `scoring/sources.csv`
(the result zips that were read, with their hashes), `status/` (the status tables of the notebooks that were read), `scoring/natives.csv` (the deposited models).

In [ ]:
def summary_tables(rows):
    cols = score_models.SCORE_COLUMNS
    write_csv(os.path.join(SCORING_DIR, "scores_long.csv"), cols, rows)
    write_csv(os.path.join(SCORING_DIR, "scores_primary.csv"), cols, [r for r in rows if str(r["primary"]) == "True"])
    write_csv(os.path.join(SCORING_DIR, "sources.csv"), ["label", "kind", "path", "bytes", "sha256", "model_files"], SOURCES)
    prim = [r for r in rows if str(r["primary"]) == "True"]
    methods = sorted({r["method"] for r in prim})
    log("\nprimary models: TM-score (-mm 1) / fragment-agnostic TM-score / Calpha match / coverage")
    log("%-5s " % "entry" + " ".join("%-34s" % m for m in methods))
    for entry in sorted({r["entry"] for r in prim}):
        cells = []
        for m in methods:
            sel = [r for r in prim if r["entry"] == entry and r["method"] == m]
            if not sel:
                cells.append("%-34s" % "-")
            else:
                r = sel[0]
                cells.append("%-34s" % ("%s / %s / %s / %s" % (r["tm_score"], r["tm_score_fns"], r["ca_match"], r["coverage"]) if r["status"] == "ok" else r["status"]))
        log("%-5s " % entry + " ".join(cells))
    states = {}
    for r in rows:
        states[r["status"]] = states.get(r["status"], 0) + 1
    log("\nstatus of the %d models: %s" % (len(rows), ", ".join("%s %d" % kv for kv in sorted(states.items()))))
    return states


if STATE["ok"] and SCORE_ROWS:
    states = summary_tables(SCORE_ROWS)
    STATE["complete"] = bool(STATE.get("phenix_used")) and all(r["status"] != "no_native" for r in SCORE_ROWS)
    if not STATE.get("phenix_used"):
        log("\nINCOMPLETE: the Phenix columns are missing (see the box above). The US-align columns and the own counts are complete.")
    if any(r["status"] == "no_native" for r in SCORE_ROWS):
        log("\nINCOMPLETE: some entries have no deposited model (status no_native).")
else:
    log("skipped: an earlier step stopped")


## Last step. Pack the results

Run this cell last, also after a problem. It works after a restart of the kernel too: it then packs the files that are on disk.

In [ ]:
import os, sys
try:
    finish
except NameError:                                  # fresh kernel: nothing of the earlier cells is left, only the files on disk
    work = None
    for cand in (os.environ.get("MICA_WORK"), "/kaggle/working/pilot", "/content/pilot", os.path.abspath("pilot")):
        if cand and os.path.isfile(os.path.join(cand, "scripts", "nbtools.py")):
            work = cand
            break
    if work is None:
        print("RECOVERY: the work folder was not found, so there is nothing to pack. Run the notebook from the top.")
    else:
        sys.path.insert(0, os.path.join(work, "scripts"))
        import nbtools
        nbtools.recover("scoring_cryozeta_d")
else:
    zip_path = finish(bool(STATE["ok"] and STATE.get("complete")), {"protocol_version": globals().get("score_models").PROTOCOL_VERSION if "score_models" in globals() else "", "usalign_commit": globals().get("USALIGN_COMMIT", ""), "usalign_version": STATE["usalign_version"], "gemmi": STATE.get("gemmi", ""), "phenix_installer": os.path.basename(globals().get("INSTALLER", "")), "phenix_installer_sha256": STATE["installer_sha256"], "phenix_version_output": STATE["phenix_version"], "phenix_used": bool(STATE.get("phenix_used")), "sources": STATE.get("sources", []), "models": len(globals().get("MODELS", [])), "workers": globals().get("WORKERS"), "input_sha256": globals().get("EXPECTED", {})})
    if STATE["ok"] and STATE.get("complete"):
        log("\nRESULT: OK. Download %s (Kaggle: the output files of this version) and send it to me." % zip_path)
    elif STATE["ok"]:
        log("\nRESULT: NOT COMPLETE (%s). Send me %s; it shows where it stopped." % (STATE.get("stopped_at") or "a step did not finish", zip_path))
    else:
        log("\nRESULT: STOPPED (%s). Send me %s." % (STATE["problem"], zip_path))
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        log("(Kaggle: open the version, then Output, then download the zip. Colab: the file list on the left.)")
